In [1]:
!pip install -q ultralytics roboflow torch torchvision --upgrade

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 40.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 80.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 126.6 MB/s eta 0:00:00


In [5]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="iyLsqEim9nAoDa24N72g")
project = rf.workspace("industrial-engineer").project("cotton-disease-zrbov")
version = project.version(20)
dataset = version.download("yolov11")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Cotton-Disease-20 in yolov11:: 100%|██████████| 23420/23420 [00:02<00:00, 8093.18it/s] 


In [2]:
import torch, torch.nn as nn
from ultralytics import YOLO
from roboflow import Roboflow

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [3]:
class CBAM(nn.Module):
    """
    CBAM that can be built from:
        CBAM()            # <-  we get this from parse_model
        CBAM(256)         # <-  or this
    """
    def __init__(self, c1=None, ratio=16, kernel_size=7):
        super().__init__()
        self._c1 = c1
        self._ratio = ratio
        self._ks = kernel_size
        self._built = False

    def _build(self, c_in):
        c_ = max(c_in // self._ratio, 16)
        # channel attention
        self.ca = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(c_in, c_, 1, bias=False), nn.ReLU(inplace=True),
            nn.Conv2d(c_, c_in, 1, bias=False))
        # spatial attention
        self.sa = nn.Conv2d(2, 1, self._ks, padding=self._ks//2, bias=False)
        self.sigmoid = nn.Sigmoid()
        self._built = True

    def forward(self, x):
        if not self._built:               # first call: finish build
            self._build(x.size(1))
        # channel attention
        ca = self.sigmoid(self.ca(x))
        x = x * ca
        # spatial attention
        avg = torch.mean(x, dim=1, keepdim=True)
        maxv = torch.amax(x, dim=1, keepdim=True)
        sa = self.sigmoid(self.sa(torch.cat([avg, maxv], 1)))
        return x * sa

In [4]:
import ultralytics.nn.modules as m
import ultralytics.nn.tasks as tasks
m.CBAM = tasks.CBAM = CBAM
print('✅ CBAM registered & parse_model-safe')

✅ CBAM registered & parse_model-safe


In [ ]:
yaml_str = """
train: /content/Cotton-Disease-20/train/images
val: /content/Cotton-Disease-20/valid/images
test: /content/Cotton-Disease-20/test/images

nc: 1
names: ['Cotton Leaf Curl Virus']

depth_multiple: 0.33
width_multiple: 0.50

anchors:
  - [10,13, 16,30, 33,23]
  - [30,61, 62,45, 59,119]
  - [116,90, 156,198, 373,326]

backbone:
  # [from, repeats, module, args]
  - [-1, 1, Conv, [64, 3, 2]]        #  0  P1/2
  - [-1, 1, Conv, [128, 3, 2]]       #  1  P2/4
  - [-1, 2, C2f, [128, False]]
  - [-1, 1, CBAM, []]                #
  - [-1, 1, Conv, [256, 3, 2]]       #  4  P3/8
  - [-1, 4, C2f, [256, False]]
  - [-1, 1, CBAM, []]                # ←
  - [-1, 1, Conv, [512, 3, 2]]       #  7  P4/16
  - [-1, 4, C2f, [512, False]]
  - [-1, 1, CBAM, []]                # ←
  - [-1, 1, Conv, [512, 3, 2]]       # 10  P5/32
  - [-1, 2, C2f, [512, False]]
  - [-1, 1, CBAM, []]                # ←
  - [-1, 1, SPPF, [512, 5]]          # 13

head:
  - [-1, 1, Conv, [256, 1, 1]]      # 14
  - [-1, 1, nn.Upsample, [None, 2, nearest]]
  - [[-1, 8], 1, Concat, [1]]       # 16
  - [-1, 2, C2f, [256, False]]
  - [-1, 1, CBAM, []]                # ←
  - [-1, 1, Conv, [128, 1, 1]]      # 19
  - [-1, 1, nn.Upsample, [None, 2, nearest]]
  - [[-1, 5], 1, Concat, [1]]       # 21
  - [-1, 2, C2f, [128, False]]
  - [-1, 1, CBAM, []]                # ←
  - [-1, 1, Conv, [128, 3, 2]]      # 24
  - [[-1, 19], 1, Concat, [1]]      # 25
  - [-1, 2, C2f, [256, False]]
  - [-1, 1, CBAM, []]                # ←
  - [-1, 1, Conv, [256, 3, 2]]      # 28
  - [[-1, 13], 1, Concat, [1]]      # 29
  - [-1, 2, C2f, [512, False]]
  - [-1, 1, CBAM, []]                # ←
  - [[23, 27, 31], 1, Detect, [1]]  # 33
"""

with open('yolo11_CBAM.yaml', 'w') as f:
    f.write(yaml_str.strip())
print('✅ CBAM yaml saved')

✅ CBAM yaml saved


In [13]:
from ultralytics import YOLO
model = YOLO('/content/yolo11_CBAM.yaml')        # builds OK
model.info()

YOLO11_CBAM summary: 173 layers, 4,022,211 parameters, 4,022,195 gradients, 13.6 GFLOPs


(173, 4022211, 4022195, 13.5511264)

In [ ]:
model.train(
    data=f"/content/yolo11_CBAM.yaml",
    epochs=50,
    imgsz=640,
    batch=64,
    name='yolov11_CBAM',
    pretrained=False,
)

Ultralytics 8.3.214 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=64, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/yolo11_CBAM.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/yolo11_CBAM.yaml, momentum=0.937, mosaic=1.0, multi_scale=False, name=yolov11_CBAM3, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0